In [1]:
import pandas as pd
import numpy as np

### Etape 1 :Reduire le perimetre en utilisant une approche metier

In [2]:
# Filterer les bonnes pays 
df2 = pd.read_csv("data/EdStatsCountry.csv")
mask = df2.Region.isnull()
false_countries = list(df2["Short Name"][mask].values)
filtered_countries = df2[~df2["Short Name"].isin(false_countries)]

In [4]:
# Fussioner le df des bonnes pays avec celui avec les indicateurs
indicators_df =pd.read_csv("data/EdStatsData.csv")
new_df = indicators_df.merge(filtered_countries, left_on = "Country Name", right_on = "Short Name", how = "inner")

In [5]:
# Enlever les colonnes non necessaires et garder que les pays, les indicateurs et les annees

years= [year for year in new_df.columns if year.isdigit()] # creer une liste des annees
new_df = new_df[["Country Name", "Indicator Name"] + years] # creer une nouvelle df avec les colonnes necessaires

####  A) Identifier la colonne qui décrit la catégorie métier à laquelle appartient chaque indicateur.

In [6]:
indicator_stats_df = pd.read_csv("data/EdStatsSeries.csv")

In [7]:
# Explorer les categories qui font sens par rapport au projet
indicator_stats_df["Topic"].value_counts()

Topic
Learning Outcomes                                                                               1046
Attainment                                                                                       733
Education Equality                                                                               426
Secondary                                                                                        256
Primary                                                                                          248
Population                                                                                       213
Tertiary                                                                                         158
Teachers                                                                                         137
Expenditures                                                                                      93
Engaging the Private Sector (SABER)                                                  

In [8]:
# creer une liste pour filterer les bonnes categories
topics_metier = ["Population","Secondary","Tertiary"]

In [9]:
# filterer le df selon les categories qui font sens
indicator_stats_df = indicator_stats_df[indicator_stats_df["Topic"].isin(topics_metier)]

# choissir les indicateurs correspondants en les mettant en minuscule
indicators = [ind.lower() for ind in indicator_stats_df["Indicator Name"]]

In [10]:
# Comparer les indicateurs dans les deux jeux de donnees 
new_df_set = set(new_df["Indicator Name"].unique())
indicator_stats_df_set = set(indicator_stats_df["Indicator Name"].unique())

common = new_df_set & indicator_stats_df_set
indicator_unique = new_df_set - indicator_stats_df_set
new_df_unique = indicator_stats_df_set - new_df_set

In [11]:
# Calculer le nombre des indicateurs restants
len(common)

575

In [12]:
# Filterer les indicateurs en gardant les taux total
excluded_terms = ["male","female","gender parity index","primary","lower"]
filtered_indicators = []

for indicator in list(common):
    has_both_sexes = "both sexes" in indicator
    has_excluded_terms = any(term in indicator for term in excluded_terms)
    if has_both_sexes and not has_excluded_terms:
        filtered_indicators.append(indicator)

In [13]:
len(filtered_indicators)

75

In [14]:
# filterer les indicateurs selon les indicateurs communs et fussioner les deux jeux des donnees
indicator_stats_df = indicator_stats_df[indicator_stats_df["Indicator Name"].isin(filtered_indicators)]
new_df = new_df.merge(indicator_stats_df[["Topic","Indicator Name"]], on = "Indicator Name", how = "inner")

####  B) Identifier les annees qui sont pertinentes

In [15]:
# Calculer le taux de valuers manquantes pour chaque annee
mask_years = (new_df[years].isnull().sum()/len(new_df)).sort_values(ascending= False)

# Filterer les annees avec 100 % des valeuers manquantes
years_to_keep = sorted([int(year) for year in mask_years.index if mask_years[year] < 1.0])


In [16]:
# Enlever les annees avec 100 % des valeurs manquantes
new_df = new_df[["Country Name","Indicator Name","Topic"] + [str(year) for year in years_to_keep]]

### Etape 2 :Reduire le perimetre en utilisant une approche data

#### A) Calculer la proportion d’indicateurs avec des valeurs renseignées

In [17]:
year_cols = [str(year) for year in years_to_keep]

In [18]:
# Creer une fonction pour calculer les valeurs renseignées par annee par indicateur
f = lambda x : x.notnull().sum()

# Aggreger le df par indicateur et creer une colonne total pour evaluer chaque indicateur
indicator_check  = new_df.groupby("Indicator Name")[year_cols].apply(f,include_groups = False)
indicator_check["total"] = indicator_check.sum(axis = 1)

In [19]:
# Verifier la distribution des valeurs total pour choissir le seuil pour enlever les indicateurs selon le data

# Calculer le coefficient de variance pour la colonne total
CV = (indicator_check["total"].std()/indicator_check["total"].mean()) * 100

# Prendre le median comme un seuil pour enlever les indicateurs avec les moins valeurs renseignées

thresh = indicator_check["total"].median()

In [20]:
# Filterer les indicateurs en comparant le taux des valeurs renseignées contre le median
filtered_indicators = indicator_check[indicator_check["total"] >= thresh]

In [21]:
new_df = new_df[new_df["Indicator Name"].isin(filtered_indicators.index)]

#### B) Calculer la proportion des annees avec des valeurs renseignées

In [22]:
# Calculer les taux des valeurs renseignées au niveau des annees
years_series = new_df[year_cols].apply(f)

# Choissir les annees avec des valeurs plus du median

years_series = years_series > years_series.median()

# Creer une liste des annees filteree

final_years = years_series[years_series == True].index

In [23]:
# Choissir les annees plus recentes
final_years = list(final_years[8:])

In [24]:
new_df = new_df[["Country Name","Indicator Name","Topic"] + final_years]


In [25]:
new_df["Indicator Name"].nunique()

38

In [26]:
# la liste des indicateurs finals

highly_relevant_indicators = [
    "Enrolment in secondary education, both sexes (number)",
    "Net enrolment rate, secondary, both sexes (%)",
    "Out-of-school youth of upper secondary school age, both sexes (number)",
    "Rate of out-of-school youth of upper secondary school age, both sexes (%)",
    "Percentage of students in secondary education enrolled in vocational programmes, both sexes (%)",
    "Enrolment in tertiary education, all programmes, both sexes (number)",
    "Gross enrolment ratio, tertiary, both sexes (%)",
    "Outbound mobility ratio, all regions, both sexes (%)",
    "Total outbound internationally mobile tertiary students studying abroad, all countries, both sexes (number)",
    "Percentage of all students in tertiary education enrolled in ISCED 5, both sexes (%)",
    "Graduates from tertiary education, both sexes (number)",
    "Gross enrolment ratio, upper secondary, both sexes (%)",
    "Lower secondary completion rate, both sexes (%)",
    "School life expectancy, secondary/tertiary, both sexes (years)",
    "Population of official school age (various levels)"    
]


In [27]:
# la creation de dataframe 
clean_df = new_df[new_df["Indicator Name"].isin(highly_relevant_indicators)]

#### C) Calculer la proportion des pays avec des valeurs renseignées

In [28]:
clean_df

,Country Name,Indicator Name,Topic,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014
3,Afghanistan,"Enrolment in secondary education, both sexes (...",Secondary,NaN,362415.00000,NaN,406895.00000,594306.00000,651453.00000,1.006841e+06,1.035782e+06,1.425009e+06,1.716190e+06,2.044157e+06,2.208963e+06,2.415884e+06,2.538420e+06,2.602734e+06
9,Afghanistan,"Enrolment in tertiary education, all programme...",Tertiary,NaN,NaN,NaN,26211.00000,27648.00000,NaN,NaN,NaN,NaN,9.518500e+04,NaN,9.750400e+04,NaN,NaN,2.628740e+05
23,Afghanistan,"Graduates from tertiary education, both sexes ...",Tertiary,NaN,NaN,NaN,NaN,NaN,NaN,6.310000e+03,6.858000e+03,9.270000e+03,9.623000e+03,NaN,NaN,NaN,NaN,2.431500e+04
25,Afghanistan,"Gross enrolment ratio, tertiary, both sexes (%)",Tertiary,NaN,NaN,NaN,1.25684,1.25603,NaN,NaN,NaN,NaN,3.903390e+00,NaN,3.755980e+00,NaN,NaN,8.662800e+00
26,Afghanistan,"Gross enrolment ratio, upper secondary, both s...",Secondary,NaN,12.21728,NaN,7.97580,11.44451,12.10556,1.696810e+01,1.755530e+01,2.180346e+01,2.605090e+01,3.213401e+01,3.986274e+01,4.456518e+01,4.626244e+01,4.390559e+01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14586,Zimbabwe,Out-of-school youth of upper secondary school ...,Secondary,740553.00000,738279.00000,774094.00000,832751.00000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.124010e+05,7.025930e+05,NaN
14587,Zimbabwe,Percentage of all students in tertiary educati...,Tertiary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.912794e+01,2.784219e+01,3.761356e+01,NaN
14601,Zimbabwe,Percentage of students in secondary education ...,Secondary,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14619,Zimbabwe,Rate of out-of-school youth of upper secondary...,Secondary,56.89821,56.19083,58.55578,62.68185,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.367036e+01,5.294437e+01,NaN


In [29]:
# Aggreger le df par pays
country_check = clean_df.groupby("Country Name")[final_years].apply(f, include_groups = False)

# Creer une colonne total pour evaluer chaque pays
country_check["Total"] = country_check.sum(axis = 1)

In [30]:
# Prendre la moyenne comme un seuil pour enlever les pays avec les moins valeurs renseignées
country_serie = country_check["Total"] > country_check["Total"].mean()

In [31]:
# Creer une liste des bonnes pays
filtered_pays  = list(country_serie[country_serie == True].index)

In [32]:
# filterer les bonnes pays
filtered_clean_df = clean_df[clean_df["Country Name"].isin(filtered_pays)]

In [33]:
filtered_clean_df

,Country Name,Indicator Name,Topic,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014
78,Albania,"Enrolment in secondary education, both sexes (...",Secondary,365664.00000,377936.000000,382205.000000,396095.000000,393503.000000,407403.00000,408980.000000,410444.000000,406553.000000,354587.000000,355871.000000,355158.000000,349269.000000,346365.000000,333291.000000
84,Albania,"Enrolment in tertiary education, all programme...",Tertiary,40125.00000,40859.000000,42160.000000,43600.000000,53014.000000,63257.00000,74747.000000,86863.000000,90606.000000,93139.000000,122326.000000,134877.000000,160839.000000,173029.000000,173819.000000
98,Albania,"Graduates from tertiary education, both sexes ...",Tertiary,4735.00000,4618.000000,5016.000000,5202.000000,5844.000000,5778.00000,7724.000000,9767.000000,15653.000000,18327.000000,22358.000000,23046.000000,NaN,30365.000000,29191.000000
100,Albania,"Gross enrolment ratio, tertiary, both sexes (%)",Tertiary,15.60373,15.974960,16.431709,16.805559,20.055540,23.31110,26.620440,30.653669,32.077801,33.400749,44.540649,49.670399,58.565491,62.547760,62.706848
101,Albania,"Gross enrolment ratio, upper secondary, both s...",Secondary,43.69709,45.356331,48.959530,51.771450,54.278069,58.65358,61.681122,67.139008,71.086517,72.855003,78.937408,86.112183,89.405228,93.003021,93.927422
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14436,West Bank and Gaza,Out-of-school youth of upper secondary school ...,Secondary,60816.00000,60371.000000,57697.000000,59290.000000,53959.000000,48277.00000,44041.000000,46026.000000,40024.000000,46765.000000,49121.000000,52697.000000,57047.000000,64160.000000,72347.000000
14437,West Bank and Gaza,Percentage of all students in tertiary educati...,Tertiary,7.24227,6.163170,5.974360,7.567400,10.077260,10.07753,10.720850,10.017540,10.519330,12.973460,14.876540,15.263610,12.909310,12.157100,11.087010
14451,West Bank and Gaza,Percentage of students in secondary education ...,Secondary,0.60707,0.706570,0.742290,0.734980,0.723630,0.80375,0.859120,0.900220,0.929800,1.149010,1.301660,0.396670,0.397340,0.385230,0.384220
14469,West Bank and Gaza,Rate of out-of-school youth of upper secondary...,Secondary,45.66109,43.661360,40.215100,40.000540,35.403840,30.74322,26.986240,26.920510,22.329590,24.921530,25.140620,26.218200,27.930260,31.016450,34.509120


In [34]:
filtered_clean_df = filtered_clean_df.copy()
filtered_clean_df["CV"] = (
    filtered_clean_df[final_years].std(axis=1) /
    filtered_clean_df[final_years].mean(axis=1)
) * 100


In [35]:
def categorize_agg_method(val):
    if val < 10 :
        return "mean"
    else:
        return "median"
filtered_clean_df["aggregation"] = filtered_clean_df["CV"].apply(categorize_agg_method)

In [36]:
# regrouper par pays + indicateur et calculer la moyenne des années
aggregated_df = filtered_clean_df.copy()
aggregated_df["Average"] = aggregated_df[final_years].mean(axis=1)

# ne garder que les colonnes utiles
pivoted = aggregated_df.pivot_table(
    index='Country Name',
    columns='Indicator Name',
    values='Average'
)



In [37]:
pivoted

Indicator Name,"Enrolment in secondary education, both sexes (number)","Enrolment in tertiary education, all programmes, both sexes (number)","Graduates from tertiary education, both sexes (number)","Gross enrolment ratio, tertiary, both sexes (%)","Gross enrolment ratio, upper secondary, both sexes (%)","Lower secondary completion rate, both sexes (%)","Net enrolment rate, secondary, both sexes (%)","Out-of-school youth of upper secondary school age, both sexes (number)","Outbound mobility ratio, all regions, both sexes (%)","Percentage of all students in tertiary education enrolled in ISCED 5, both sexes (%)","Percentage of students in secondary education enrolled in vocational programmes, both sexes (%)","Rate of out-of-school youth of upper secondary school age, both sexes (%)","Total outbound internationally mobile tertiary students studying abroad, all countries, both sexes (number)"
Country Name,,,,,,,,,,,,,
Albania,3.762216e+05,9.288400e+04,1.340171e+04,33.931094,67.790864,90.424578,76.275414,7.799525e+04,20.755071,1.232904,5.976005,35.310615,17228.357143
Argentina,4.058737e+06,2.309615e+06,2.196569e+05,68.846463,75.139845,83.358775,81.952727,3.469044e+05,0.381327,28.938100,NaN,17.047550,8366.357143
Armenia,3.287558e+05,1.200137e+05,2.735992e+04,41.949221,88.008294,92.372573,87.369760,2.523625e+04,3.528389,19.905299,4.459767,20.961655,4354.571429
Aruba,7.186077e+03,1.927667e+03,2.890000e+02,30.135898,88.933993,94.086675,75.659721,2.852222e+02,8.248490,66.114070,16.498523,6.681060,160.642857
Australia,2.383824e+06,1.134004e+06,3.015097e+05,76.502040,193.911774,NaN,86.076093,7.079533e+04,0.801196,17.546985,37.257383,12.282437,8986.357143
...,...,...,...,...,...,...,...,...,...,...,...,...,...
United Kingdom,5.541159e+06,2.320334e+06,6.599709e+05,58.954296,103.333174,NaN,94.064020,2.404553e+05,1.111149,22.539829,18.960337,7.720763,25674.071429
United States,2.404698e+07,1.788125e+07,2.801773e+06,84.056566,88.997697,NaN,88.731939,1.307216e+06,0.308126,24.061673,NaN,10.313555,54303.500000
Uruguay,3.122505e+05,1.273120e+05,8.425500e+03,50.171839,87.669744,70.155420,71.969772,3.324100e+04,1.703454,17.277847,16.614207,20.825718,2182.785714


In [38]:
pivoted.to_csv("resultats.csv", index = True)